# Fundamentals of Machine Learning — Exercise 7
## Boosting: hands-on examples and library implementations

**Format:** independent work, approximately **70–80 minutes**.

### 🎯 Aim of the exercise

In this exercise, you will:

- build a small **gradient boosting regressor** from scratch,
- perform several rounds of **AdaBoost** and observe changing sample weights,
- compare a Decision Tree ensemble baseline with **AdaBoost**, **Gradient Boosting**, and **Histogram Gradient Boosting**,
- investigate the interaction between the learning rate and the number of trees,
- identify overfitting and use **early stopping**.  

Tasks marked **⭐** are optional.


### How to work
1. Read the text and **predict** the result before you run a cell.
2. Complete the `TODO`s (they are short: 1–3 lines). Variables set to `None` are waiting for you.
3. Run the **check** cell below the task: ✅ means correct, ❌ gives a hint what to look at.
4. Stuck for more than 5 minutes? Open the 💡 hint. Still stuck? Continue and write the problem down in the last section.
5. Write short answers into the ✍️ cells. Run the notebook **top to bottom** (cells depend on each other).

The optional extension at the end is not required.

## 0. Setup

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingRegressor,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier)
from sklearn.metrics import (log_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay)

plt.rcParams["axes.grid"] = True
RANDOM_STATE = 42


def check(test, ok="Correct.", hint="Not yet."):
    # Run a small self-check without stopping the notebook.
    try:
        passed = bool(test())
    except Exception as error:
        passed = False
        hint = hint + f"  [{type(error).__name__}: {error}]"

    print(("✅ " + ok) if passed else ("❌ " + hint))

## 1. Theory in a Nutshell

### 1.1 Bagging and boosting

| | Bagging / Random Forest | Boosting |
|---|---|---|
| trees are built | independently and can be trained in parallel | sequentially; each new learner reacts to the current errors |
| typical trees | relatively deep | usually shallow weak learners |
| main effect | averaging mainly reduces variance | sequential correction mainly reduces bias |
| adding trees | usually stabilizes performance | may eventually fit noise and worsen validation performance |

### 1.2 Gradient boosting for regression

With squared-error loss, begin with a constant prediction and repeatedly fit a small tree to the residuals:

$$
F_0=\bar y, \qquad
r_i=y_i-F_{m-1}(x_i), \qquad
F_m(x)=F_{m-1}(x)+\nu h_m(x).
$$

The learning rate $\nu$ controls how much each new tree contributes. For squared error, the residual is also the negative gradient of the loss with respect to the current prediction. Gradient boosting therefore performs gradient-descent steps in the space of functions.

### 1.3 AdaBoost for classification

AdaBoost changes the weights of observations. Misclassified observations receive greater weight, so the next weak learner focuses more strongly on them.

$$
\varepsilon_m=\sum_{i:h_m(x_i)\ne y_i}w_i,
\qquad
\alpha_m=\frac12\ln\frac{1-\varepsilon_m}{\varepsilon_m}.
$$

A learner with a small weighted error receives a larger vote $\alpha_m$.

### 1.4 Histogram gradient boosting

Classical tree construction considers many possible thresholds. Histogram-based algorithms first place continuous values into a limited number of bins and evaluate splits at bin boundaries. This usually makes training much faster on large datasets, while the approximation often loses little predictive performance.

**Demo 1 — boosting in action.** Noisy sine curve, 50 stumps (trees with a single split), learning rate 0.3. Watch how the sum of stumps gradually draws the curve (just run it).

In [ ]:
rng = np.random.default_rng(0)
x = np.sort(rng.uniform(0, 10, 60)).reshape(-1, 1)
y = np.sin(x).ravel() + rng.normal(0, 0.25, 60)

grid = np.linspace(0, 10, 300).reshape(-1, 1)
F_train = np.full(60, y.mean())        # current prediction at the training points
F_grid = np.full(300, y.mean())        # current prediction on the grid (for drawing)

fig, axs = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
panels = [1, 3, 10, 50]
for m in range(1, 51):
    residual = y - F_train
    stump = DecisionTreeRegressor(max_depth=1).fit(x, residual)
    F_train = F_train + 0.3 * stump.predict(x)
    F_grid = F_grid + 0.3 * stump.predict(grid)
    if m in panels:
        ax = axs[panels.index(m)]
        ax.scatter(x, y, s=15, color="gray")
        ax.plot(grid, F_grid, color="C3", lw=2)
        ax.set_title(f"after {m} stump(s)")
plt.tight_layout()
plt.show()

❓ After 1 stump the prediction has only 2 levels. Why does the sum of 50 stumps give a smooth-looking curve?

✍️ **Your answer:** *(double-click this cell and write here)*

## 2. Gradient boosting by hand 

Ten points in one dimension:

In [ ]:
X_h = np.arange(1, 11, dtype=float).reshape(-1, 1)
y_h = np.array([1.2, 1.9, 3.1, 3.8, 5.2, 5.0, 3.9, 3.1, 2.2, 1.4])

plt.figure(figsize=(6, 3.5))
plt.scatter(X_h, y_h, s=60, color="C0", edgecolor="k")
plt.xlabel("x"); plt.ylabel("y"); plt.title("A hill that no single stump can describe")
plt.show()

### 2.1 Worked first step
1. $F_0=\bar y = 30.8/10 = 3.08$ (a horizontal line), training MSE = 1.79.
2. Residuals $r_i=y_i-3.08$: `[-1.88, -1.18, 0.02, 0.72, 2.12, 1.92, 0.82, 0.02, -0.88, -1.68]`.
3. The best stump on the residuals splits at $x\le 2.5$: it predicts $-1.53$ on the left (mean of the first two residuals) and $+0.38$ on the right.
4. With $\nu=0.5$: $F_1=F_0+0.5\,h_1$, MSE drops from 1.79 to 1.35.

### 2.2 Task: boosting loop
Complete the three lines inside the loop (everything else is given).

<details><summary>💡 Hint</summary>

1) `residual = y - F`  2) `tree = DecisionTreeRegressor(max_depth=1).fit(X, residual)`  3) `F = F + lr * tree.predict(X)`

</details>

In [ ]:
def gb_fit(X, y, n_trees=30, lr=0.3):
    F0 = y.mean()                        # start: a constant prediction
    F = np.full(len(y), F0)              # current predictions at the training points
    trees, mses = [], [np.mean((y - F) ** 2)]
    for m in range(n_trees):
        residual = None                  # TODO 1: what is still wrong = y - F
        tree = None                      # TODO 2: a stump (max_depth=1) fitted to the residuals
        F = F                            # TODO 3: add lr times the stump's prediction
        trees.append(tree)
        mses.append(np.mean((y - F) ** 2))
    return F0, trees, mses


def gb_predict(F0, trees, X, lr=0.3):
    F = np.full(len(X), F0)
    for tree in trees:
        F = F + lr * tree.predict(X)
    return F

In [ ]:
F0, trees, mses = gb_fit(X_h, y_h, n_trees=30, lr=0.3)
print("MSE after 0, 1, 2, 3, 10, 30 trees:", [round(mses[i], 3) for i in [0, 1, 2, 3, 10, 30]])

sk = GradientBoostingRegressor(n_estimators=30, learning_rate=0.3, max_depth=1,
                               criterion="squared_error", random_state=0).fit(X_h, y_h)

check(lambda: np.isclose(mses[0], 1.79, atol=0.01), "MSE at the start is 1.79 (constant = mean)", "The start MSE should be 1.79.")
check(lambda: all(mses[i + 1] <= mses[i] + 1e-12 for i in range(30)), "MSE never increases", "Each step must not make the training MSE worse.")
check(lambda: np.isclose(mses[1], 1.491, atol=1e-3), "after one tree (lr = 0.3) MSE = 1.491", "Expected 1.491 after the first tree.")
check(lambda: np.allclose(gb_predict(F0, trees, X_h), sk.predict(X_h)), "your predictions are identical to sklearn's GradientBoostingRegressor", "Should equal sklearn's predictions.")

Your boosting after 1, 3, 10 and 30 trees (just run it):

In [ ]:
gx = np.linspace(0.5, 10.5, 200).reshape(-1, 1)
fig, axs = plt.subplots(1, 4, figsize=(16, 3.3), sharey=True)
for ax, k in zip(axs, [1, 3, 10, 30]):
    ax.scatter(X_h, y_h, color="C0", edgecolor="k", zorder=3)
    ax.plot(gx, gb_predict(F0, trees[:k], gx), color="C3", lw=2)
    ax.set_title(f"{k} tree(s), MSE = {mses[k]:.2f}")
plt.tight_layout()
plt.show()

❓ The first stump splits at $x\le 2.5$. Why there, and not in the middle of the hill?

✍️ **Your answer:** *(double-click this cell and write here)*

### 2.3 The learning rate (just run it)
The same boosting with three learning rates. *(On 10 training points a larger step always fits faster — the point of a small step is the behaviour on **new** data, see Part 5.)*

In [ ]:
plt.figure(figsize=(6.5, 3.8))
for lr in [0.05, 0.3, 1.0]:
    _, _, m_lr = gb_fit(X_h, y_h, n_trees=60, lr=lr)
    plt.plot(m_lr, label=f"learning rate {lr}")
plt.xlabel("number of trees"); plt.ylabel("training MSE"); plt.yscale("log"); plt.legend(); plt.show()

❓ Which learning rate fits the training data fastest? Why would you still prefer a small learning rate with many trees in practice?

✍️ **Your answer:** *(double-click this cell and write here)*

## 3. AdaBoost by hand 
Ten points in 2D, labels $-1$ (blue) and $+1$ (red). A single stump (one vertical or horizontal line) cannot classify them all.

In [ ]:
X_a = np.array([[8, 1], [3, 2], [4, 9], [2, 2], [4, 8], [1, 8], [8, 4], [1, 5], [7, 8], [2, 7]], dtype=float)
y_a = np.array([1, 1, 1, 1, 1, -1, 1, -1, -1, -1])

plt.figure(figsize=(5, 4.5))
plt.scatter(X_a[:, 0], X_a[:, 1], c=y_a, cmap="coolwarm", s=90, edgecolor="k")
for i, (a, b) in enumerate(X_a):
    plt.annotate(str(i), (a, b), textcoords="offset points", xytext=(7, 5))
plt.xlabel("x1"); plt.ylabel("x2"); plt.title("10 points, label +1 = red, -1 = blue")
plt.show()

### 3.1 Task: one round of AdaBoost
`adaboost_round(X, y, w)` fits a stump to the **weighted** data and returns the new weights. The stump and its predictions `h` are given. Write the three formulas from section 1.3.

<details><summary>💡 Hint</summary>

1) `eps = w[h != y].sum()` (the weights of the wrong samples)  2) `alpha = 0.5 * np.log((1 - eps) / eps)`  3) `w_new = w * np.exp(-alpha * y * h)` and then `w_new = w_new / w_new.sum()`

</details>

In [ ]:
def adaboost_round(X, y, w):
    stump = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X, y, sample_weight=w)
    h = stump.predict(X)                 # predictions, each -1 or +1
    eps = None                           # TODO 1: weighted error = sum of the weights of misclassified samples
    alpha = None                         # TODO 2: 0.5 * ln((1 - eps) / eps)
    w_new = None                         # TODO 3: w * exp(-alpha * y * h), then divide by the sum
    return stump, eps, alpha, w_new

In [ ]:
w = np.full(10, 1 / 10)                  # start: all samples equally important
rounds = []
for m in range(3):
    stump, eps, alpha, w_new = adaboost_round(X_a, y_a, w)
    rounds.append((stump, eps, alpha, w_new))
    print(f"round {m + 1}: split on x{stump.tree_.feature[0] + 1} <= {stump.tree_.threshold[0]},  error = {eps:.3f},  alpha = {alpha:.3f}")
    w = w_new

def ada_predict(rounds, X):
    total = np.zeros(len(X))
    for stump, eps, alpha, w_new in rounds:
        total = total + alpha * stump.predict(X)
    return np.sign(total)

acc1 = np.mean(rounds[0][0].predict(X_a) == y_a)
acc3 = np.mean(ada_predict(rounds, X_a) == y_a)
print(f"accuracy of the first stump alone: {acc1:.2f},   of the 3 stumps together: {acc3:.2f}")

In [ ]:
check(lambda: np.isclose(rounds[0][1], 0.2) and np.isclose(rounds[0][2], 0.693, atol=1e-3), "round 1: error 0.2, alpha 0.693", "In round 1 two of ten samples (weight 0.1 each) are wrong: error 0.2, alpha = 0.5*ln(4) = 0.693.")
check(lambda: np.isclose(rounds[0][3].sum(), 1.0) and np.isclose(rounds[0][3].max(), 0.25), "weights are normalised; the 2 wrong samples now have weight 0.25 each", "After round 1 the two misclassified samples should have weight 0.25 each and all weights sum to 1.")
check(lambda: np.isclose(rounds[1][1], 0.125, atol=1e-3) and np.isclose(rounds[2][1], 0.179, atol=1e-3), "errors of rounds 2 and 3: 0.125 and 0.179", "Expected errors 0.200, 0.125, 0.179.")
check(lambda: acc3 == 1.0 and acc1 == 0.8, "3 weak stumps classify all 10 points correctly, one stump only 8 of 10", "Expected 0.8 for one stump and 1.0 for three.")

ada = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=3, random_state=0).fit(X_a, y_a)
print("sklearn AdaBoost errors:", np.round(ada.estimator_errors_, 3), "  its stump weights:", np.round(ada.estimator_weights_, 3), " (yours x 2:", np.round([2 * r[2] for r in rounds], 3), ")")

Weights of the samples before round 1, 2 and 3 (marker size = weight; just run it):

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 4.2))
w_before = np.full(10, 0.1)
for m, ax in enumerate(axs):
    ax.scatter(X_a[:, 0], X_a[:, 1], c=y_a, cmap="coolwarm", s=2500 * w_before, edgecolor="k")
    st = rounds[m][0]
    f, t = st.tree_.feature[0], st.tree_.threshold[0]
    (ax.axvline if f == 0 else ax.axhline)(t, color="k", lw=2)
    ax.set_title(f"round {m + 1}: error = {rounds[m][1]:.3f}, alpha = {rounds[m][2]:.2f}")
    ax.set_xlim(0, 10); ax.set_ylim(0, 10)
    w_before = rounds[m][3]
plt.tight_layout()
plt.show()

❓ (a) Why do the wrong samples get a bigger weight? (b) After round 1 the two wrong samples have weight 0.25 each, i.e. **together 0.5** — is that a coincidence?

✍️ **Your answer:** *(double-click this cell and write here)*

## 4. Compare Boosting Implementations

We will reuse the Breast Cancer Wisconsin dataset from the previous exercises. Each row represents one tumour sample.

For consistency:

- `1` = malignant tumour,
- `0` = benign tumour.

We compare:

- Random Forest as a bagging baseline,
- AdaBoost with decision stumps,
- classical Gradient Boosting,
- Histogram Gradient Boosting.

The models are evaluated on the same test observations. Scaling is not required because tree splits compare feature values with thresholds rather than calculating distances.

In [ ]:
cancer = load_breast_cancer(as_frame=True)

X = cancer.data.copy()
y = (cancer.target == 0).astype(int)
y.name = "malignant"

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Malignant proportion in training data:", round(y_train.mean(), 3))

### 4.1 Task: train and evaluate the models

Complete the two TODOs inside the loop.

<details><summary>💡 Hint</summary>

Use `model.fit(X_train, y_train)` and `y_pred = model.predict(X_test)`.

</details>

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "AdaBoost": AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=100,
        learning_rate=0.5,
        random_state=RANDOM_STATE,
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=2,
        random_state=RANDOM_STATE,
    ),
    "Histogram Gradient Boosting": HistGradientBoostingClassifier(
        max_iter=100,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
    ),
}

rows = []
fitted_models = {}

for name, model in models.items():
    start = time.perf_counter()

    # TODO 1: fit the model

    fit_time = time.perf_counter() - start

    y_pred = None  # TODO 2: predict the test labels
    fitted_models[name] = model

    rows.append({
        "Model": name,
        "Train accuracy": model.score(X_train, y_train),
        "Test accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "Fit time [s]": fit_time,
    })

results = pd.DataFrame(rows).set_index("Model")
results.round(3)

In [ ]:
check(lambda: results.shape == (4, 6),
    "all four models were evaluated",
    "The results table should contain four rows and six columns.")
check(lambda: results["Test accuracy"].between(0.80, 1.0).all(),
    "all models have plausible test accuracy",
    "Check that predictions were made on X_test and compared with y_test.")
check(lambda: results["Recall"].between(0, 1).all(),
    "recall values are valid",
    "Recall must be between 0 and 1.")

In [ ]:
results[["Test accuracy", "Recall", "F1"]].plot(
    kind="bar",
    figsize=(9, 4),
    ylim=(0.7, 1.0))
plt.ylabel("Score")
plt.title("Model Comparison on the Test Set")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

❓ Which model has the highest test accuracy and which the highest recall for malignant tumours? By how many test patients does it beat the worst model? (The test set has 114 samples, so one patient is about 0.9 percentage points.)

❓ Does boosting automatically outperform Random Forest on this dataset? Can you decide that from a single train–test split?

❓ Why is the training time measured here only an approximate observation rather than a universal property of the algorithms?

❓ Why would recall be particularly important when class `1` represents a malignant tumour?

✍️ **Your answer:** *(double-click this cell and write here)*

### 4.2 Inspect the errors of the model with the highest recall

The following code selects the model automatically.

In [ ]:
best_recall_name = results["Recall"].idxmax()
best_recall_model = fitted_models[best_recall_name]
best_recall_predictions = best_recall_model.predict(X_test)

print("Highest-recall model:", best_recall_name)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_recall_predictions,
    display_labels=["benign", "malignant"],
    cmap="Blues")
plt.title(best_recall_name)
plt.show()

❓ How many malignant tumours were predicted as benign? Which cell represents these false negatives?

✍️ **Your answer:** *(double-click this cell and write here)*

### 4.3 ⭐ Is the difference real?

One train–test split gives one number per model. Repeat the experiment on 10 different splits and look at the mean and the spread (standard deviation).

In [ ]:
from sklearn.base import clone

n_repeats = 10
scores = {name: [] for name in models}

for seed in range(n_repeats):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
    for name, model in models.items():
        m = clone(model).fit(X_tr, y_tr)
        scores[name].append(accuracy_score(y_te, m.predict(X_te)))

summary = pd.DataFrame({
    "mean accuracy": {name: np.mean(v) for name, v in scores.items()},
    "std": {name: np.std(v) for name, v in scores.items()},
})
display(summary.round(3))

plt.figure(figsize=(8, 4))
plt.boxplot([scores[name] for name in models])
plt.xticks(range(1, len(models) + 1), list(models), rotation=15)
plt.ylabel("test accuracy")
plt.title(f"Accuracy over {n_repeats} different train-test splits")
plt.tight_layout()
plt.show()

❓ Compare the differences between the mean accuracies with the standard deviation. Can you say that one model is clearly the best? What would you need to be more sure?

## 5. Overfitting, Learning Rate, and Early Stopping

Boosting repeatedly focuses on what remains unexplained. After useful structure has been learned, later trees may begin fitting noise. Training loss can therefore continue decreasing while validation or test loss starts increasing.

In contrast, adding trees to a Random Forest usually stabilizes the averaged prediction rather than creating the same progressive error-correction behaviour.

In [ ]:
Xn, yn = make_classification(n_samples=4000, n_features=20, n_informative=6, n_redundant=2,
                             flip_y=0.15, random_state=0)
Xn_tr, Xn_te, yn_tr, yn_te = train_test_split(Xn, yn, test_size=0.4, random_state=0, stratify=yn)

model = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, early_stopping=False, random_state=0)
model.fit(Xn_tr, yn_tr)

train_loss = [log_loss(yn_tr, p) for p in model.staged_predict_proba(Xn_tr)]    # loss after 1, 2, ..., 300 trees
print(len(train_loss), "values computed on the training set")

### 5.1 Task: the test curve and the best number of trees
`staged_predict_proba(X)` gives the predicted probabilities after 1, 2, 3, ... trees. Compute the **log-loss on the test data** after every tree (as for the training data above) and find the iteration with the lowest test loss.

<details><summary>💡 Hint</summary>

`test_loss = [log_loss(yn_te, p) for p in model.staged_predict_proba(Xn_te)]`, then `best_iter = np.argmin(test_loss) + 1` (+1 because the list starts with 1 tree = index 0).

</details>

In [ ]:
test_loss = None                         # TODO: log-loss on the TEST data after every tree
best_iter = None                         # TODO: number of trees with the lowest test loss
print("best number of trees:", best_iter)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(train_loss, label="training loss")
plt.plot(test_loss, label="test loss")
plt.axvline(best_iter, color="gray", ls="--", label=f"best = {best_iter} trees")
plt.xlabel("number of trees"); plt.ylabel("log-loss"); plt.legend(); plt.show()

check(lambda: len(test_loss) == 300 and 15 <= best_iter <= 100, "the test loss is lowest after a few tens of trees", "The lowest test loss should be somewhere around 40 trees.")
check(lambda: test_loss[-1] > min(test_loss) + 0.1 and train_loss[-1] < 0.05, "later trees fit only noise: the training loss → 0, the test loss rises again", "Expected: the final test loss clearly higher than the minimum, the training loss near 0.")

❓ The training loss keeps decreasing, the test loss rises again. What is happening? Compare with the random forest from the last exercise.

✍️ **Your answer:** *(double-click this cell and write here)*

### 5.2 Learning rate

A smaller learning rate makes every update less aggressive and generally requires more trees. It may improve generalization, but it does not guarantee a better model: the learning rate and number of trees must be selected together.

Run the comparison below.

In [ ]:
plt.figure(figsize=(7, 4))
for lr in [0.3, 0.1, 0.03]:
    m = HistGradientBoostingClassifier(max_iter=300, learning_rate=lr, early_stopping=False, random_state=0).fit(Xn_tr, yn_tr)
    loss = [log_loss(yn_te, p) for p in m.staged_predict_proba(Xn_te)]
    plt.plot(loss, label=f"learning rate {lr}  (best: {int(np.argmin(loss)) + 1} trees, loss {min(loss):.3f})")
plt.xlabel("number of trees"); plt.ylabel("test log-loss"); plt.ylim(0.35, 1.1); plt.legend(); plt.show()

❓ How do the best number of trees and the shape of the curve change with the learning rate? What is the rule of thumb for choosing the learning rate and the number of trees?

✍️ **Your answer:** *(double-click this cell and write here)*

### 5.3 Task: early stopping
Instead of choosing the number of trees by hand, the model sets aside a **validation part of the training data** and stops when the validation loss has not improved for `n_iter_no_change` iterations. Fit the model below on the training data (the parameters are given) and compare its test accuracy with the model that trained all 300 trees.

*(The test set must not be used for stopping — otherwise it is no longer an honest estimate. Same principle as in Exercises 5 and 6.)*

<details><summary>💡 Hint</summary>

`early.fit(Xn_tr, yn_tr)`; then `early.n_iter_` is the number of trees that were used and `early.score(Xn_te, yn_te)` the test accuracy.

</details>

In [ ]:
early = HistGradientBoostingClassifier(max_iter=1000, learning_rate=0.1, early_stopping=True,
                                       validation_fraction=0.15, n_iter_no_change=10, random_state=0)
# TODO: fit `early` on the training data (Xn_tr, yn_tr)

print("trees used:", early.n_iter_)
print("test accuracy, early stopping:", round(early.score(Xn_te, yn_te), 3))
print("test accuracy, all 300 trees :", round(model.score(Xn_te, yn_te), 3))

In [ ]:
check(lambda: early.n_iter_ < 150, "early stopping ended long before the 1000 allowed iterations (it used about 50 trees)", "Early stopping should stop after roughly 50 trees.")
check(lambda: early.score(Xn_te, yn_te) >= model.score(Xn_te, yn_te) - 0.01, "with far fewer trees it is at least as good as the over-trained model", "Early stopping should not be (much) worse than the 300-tree model.")

❓ (a) Why can we not simply use the test set for stopping? (b) LightGBM has the same idea: `fit(X, y, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(10)])`. What is the validation set there?

✍️ **Your answer:** *(double-click this cell and write here)*

## 6. Wrap-up

### Summary table

Complete the table using your results and the explanations above.

| | Random Forest | AdaBoost | Gradient Boosting | Histogram Gradient Boosting |
|---|---|---|---|---|
| how are learners built? | | | | |
| what does the next tree react to? | | | | |
| key complexity control | | | | |
| can sequential training fit noise? | | | | |
| main practical advantage | | | | |

### Final questions

1. Why are weak learners in boosting usually shallow?
2. What is the “gradient” in gradient boosting?
3. How is AdaBoost different from fitting trees to residuals?
4. Why must early stopping use validation data rather than test data?
5. Why can we not conclude from one train–test split that one algorithm is universally best?

## ⭐ Optional Extension

Try one controlled change:

- change `max_depth` in `GradientBoostingClassifier`, or
- change the AdaBoost stump to `max_depth=2`, or
- change `learning_rate` and compensate by changing the number of estimators.

Compare training accuracy, test accuracy, recall, and fitting time with the original model. Explain the effect of the change rather than searching for a perfect score.